# Deployment Runbook — Dev Environment (Steps 3–4)

**Framework:** File Validation & Routing (`fvr`) • **Purpose:** mechanical bring-up of the dev stack — MongoDB (metadata) + FastAPI upload API + validation/routing worker (bytes on UC volumes) • **Status (2026-11-09):** all artifacts authored and reviewed; **MongoDB is NOT yet provisioned** (no Mongo connection exists in this workspace — verified against Unity Catalog connections), so this runbook is executed on infrastructure outside Databricks: an Atlas cluster and a small container host (Docker Compose or k8s) for the two services.

## 1. Already done (do not redo)

| Item | State |
| --- | --- |
| UC volumes `mdd_platform.fvr.{staging,intake,quarantine}` | ✅ created, roots seeded in `volume_map`, smoke-tested |
| Registry + operational DDL & seeds (Steps 1–2, mongosh) | ✅ authored, re-runnable, real schema contracts |
| Upload API code (Step 3) | ✅ authored + reviewed (F1–F4 fixed) |
| Validation/routing worker code (Step 4) | ✅ authored + reviewed (V1–V6 fixed) |
| R2 fixtures (`fvr_test_files/`) + dry-run matrix | ✅ generated, headers verified; **pre-R2 simulation 11/11 PASS on real volumes** (see R2 notebook execution log) |

## 2. Bring-up sequence (each step is idempotent — safe to re-run)

| # | Step | Where | Verify |
| --- | --- | --- | --- |
| 1 | **Atlas cluster** (M10 dev tier, replica set, region near the API host) + DB user `fvr_api` (readWrite on `fvr`) | Atlas console | `mongosh "mongodb+srv://…" --eval 'rs.status().ok'` → 1; network access allowlisted for the API host |
| 2 | **Apply Step 1** (registry DDL + seeds) then **Step 2** (operational DDL + index guards) | `mongosh` from any allowlisted host | both notebooks' verification cells: 7 collections seeded 1/5/5/8/3/7, all 8 ops collections `validationAction: error`, 0 TTL indexes |
| 3 | **Service principal** `svc-fvr-dev` in Databricks; generate OAuth secret (M2M) | Databricks admin | token exchange curl returns an access token |
| 4 | **Grants** — dev: one SP is fine: `WRITE FILES` on `mdd_platform.fvr.staging` (API), `READ FILES` + `WRITE FILES` on staging/intake/quarantine (worker). Prod: split into two SPs | UC grants | API smoke PUT succeeds; worker can read staging + write intake |
| 5 | **Deploy API** — `uvicorn fvr_api.main:app` behind gunicorn (2 pods), env per cell 3, `/healthz` + `/metrics` | container host | `curl $API/healthz` → 200; startup cross-check (volume_map root vs settings) passes |
| 6 | **Deploy worker** — `python -m worker.main` (1 pod, dev), same env + `WORKER_ID`; `/metrics` | container host | `event_backlog` metric appears; no claim errors in logs |
| 7 | **Step-3 smoke** — manifest → stream → complete (cell 6 of the Step-3 notebook) against dev | any host with curl | `COMPLETED`, `file_registry` RECEIVED, exactly one `FILE_ARRIVED`, staging copy present |
| 8 | **Step-4 smoke** — watch the file flow `RECEIVED → ROUTED_INTAKE`, 8 PASS rows, staging dir gone | mongosh | worker pass criteria cell in Step-4 notebook |
| 9 | **R2 full execution** — cases 1–12 (incl. the kill -9 chaos case) + locust S1–S5, record verdict | locust host | R2 notebook sign-off table |

## 3. Environment & secrets (single source)

| Variable | Value (dev) | Notes |
| --- | --- | --- |
| `DATABRICKS_HOST` | `https://dbc-16f873ab-edf2.cloud.databricks.com` | this workspace |
| `DATABRICKS_CLIENT_ID` / `DATABRICKS_CLIENT_SECRET` | svc-fvr-dev OAuth M2M | from your secret manager — **never** in the repo; dev fallback: a PAT in `DATABRICKS_TOKEN` |
| `MONGO_URI` | `mongodb+srv://fvr-api:<pw>@<cluster>…/fvr?retryWrites=true` | Atlas; user has readWrite on `fvr` only |
| `STAGING_ROOT` | `/Volumes/mdd_platform/fvr/staging` | cross-checked against `volume_map` at startup — mismatch = 503 fail-fast |

**Rollback:** services are stateless — `docker compose down` leaves only Mongo + volume bytes; a bad deploy is rolled back by redeploying the previous image tag. Mongo DDL is guarded/idempotent (Steps 1–2 re-runnable); a hard reset for dev = drop the `fvr` database and re-run Steps 1–2 — **never** drop the UC volumes (they hold intake bytes; re-create only per the Purge Job rules).

**Common failure modes:** `401/403` on volume PUT → SP missing `WRITE FILES` or wrong host; Mongo `TransactionalNotSupported` → single-node Atlas without replica set (use M10+ or a replica-set M0); startup 503 → `volume_map` roots vs `STAGING_ROOT` drift; worker claims nothing → check the V1 `$or` claim filter (a regression reintroducing a bare `$lt` starves all fresh events).

## 4. After dev is up

* **R2 full run** (step 9) closes Phase-1 validation; record the verdict + issues log in the R2 notebook.
* **Prod deltas** (not dev): two separate SPs (API: staging only; worker: staging-read + intake/quarantine-write); HA Atlas; gunicorn workers sized from S3 numbers; alerts wired to the `notifications` seeds; environment-specific volumes (open item #5 of the Storage Addendum) — `volume_map` is env-scoped data, not code.